# MAVE-Q v4 (`VetQuant-RSI`) — Train Open-Source Model (4-bit QLoRA SFT + DPO) & Push to Hugging Face

> **Nền tảng Cloud Miễn phí ($0)**: Google Colab Free (`1x T4 16GB`), Kaggle Free (`2x T4 32GB`, 30h/tuần), Kaggle TPU (`v5e-8`), Lightning AI / Modal.  
> **Peak VRAM**: `< 4.5 GB VRAM` cho trọn bộ **MAVE-Q v4 Core (`~10M` params)** + **Open-Source LLM/SLM (`Qwen/Qwen3-1.7B`, `Qwen2.5-1.5B-Instruct`, hoặc `Qwen3-4B`)** với 2-Stage 4-bit NF4 QLoRA SFT -> DPO, tự động dọn VRAM giữa các stage, hỗ trợ `--resume` và **Push thẳng lên Hugging Face Hub của bạn** sau khi train xong!

### Cách chuẩn bị Hugging Face Token (An toàn, không lộ key):
- **Trên Google Colab**: Bấm biểu tượng chìa khóa **🔑 (Secrets)** ở thanh bên trái -> Thêm tên `HF_TOKEN` -> Dán token có quyền **Write** (`https://huggingface.co/settings/tokens`) -> Bật **Notebook access**.
- **Trên Kaggle**: Menu **Add-ons -> Secrets** -> Thêm `HF_TOKEN` -> Tích chọn ô đính kèm vào Notebook.
- *(Nếu chưa cài Secret, Cell 6 sẽ hiện ô nhập ẩn `getpass` an toàn ngay khi chạy.)*

In [ ]:
# Cell 1: Kiểm tra phần cứng GPU / TPU / VRAM trên Colab hoặc Kaggle
import os

platform = 'Kaggle' if os.path.exists('/kaggle/working') else ('Colab' if os.path.exists('/content') else 'Local')
print('Detected Platform:', platform)
try:
    import torch

    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        bf16_ok = bool(hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported())
        print(f'  GPU {i}: {p.name} ({p.total_memory / 1024**3:.2f} GB VRAM, bf16={bf16_ok})')
except Exception as e:
    print('PyTorch check:', e)
if os.environ.get('TPU_NAME') or os.environ.get('PJRT_DEVICE') == 'TPU':
    print('TPU v5e-8 detected! Using torch_xla BF16 path.')

In [ ]:
# Cell 2: Clone / Pull Repository VN_Trading_Agents
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/KenyaWashed/VN_Trading_Agents.git'
REPO_DIR = 'VN_Trading_Agents'

if not Path('train_colab_kaggle.py').exists():
    if not Path(REPO_DIR).exists():
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=False)
head_sha = subprocess.check_output(['git', 'log', '-n', '1', '--oneline'], text=True).strip()
print('Working dir:', Path.cwd())
print('Latest git commit:', head_sha)
if not Path('src/vn_rsi/veteran/hf_hub_pusher.py').exists():
    print('⚠️ CẢNH BÁO: Repo trên GitHub đang chưa có bản cập nhật mới nhất (thiếu hf_hub_pusher.py).')
    print('👉 Hãy commit & `git push origin main` từ máy local của bạn lên GitHub rồi chạy lại Cell 2!')

In [ ]:
# Cell 3: Cài đặt thư viện chuẩn (Giữ nguyên CUDA PyTorch gốc của Colab/Kaggle + Nâng cấp đồng bộ TRL/PEFT/Transformers/HF Hub)
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

# 1. Chỉ cài các gói cơ sở còn thiếu (tránh nâng cấp đụng pyarrow/cudf có sẵn) & cài editable repo (--no-deps giữ nguyên CUDA torch)
base_pkgs = [
    ('pydantic', 'pydantic'),
    ('yaml', 'pyyaml'),
    ('pyarrow', 'pyarrow'),
    ('scipy', 'scipy'),
    ('sklearn', 'scikit-learn'),
    ('httpx', 'httpx'),
    ('pytest', 'pytest'),
]
missing_base = [pip_name for mod_name, pip_name in base_pkgs if importlib.util.find_spec(mod_name) is None]
if missing_base:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_base])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'])

# 2. Nâng cấp (-U) bộ 4-bit QLoRA + DPO + Hugging Face Hub để hỗ trợ Qwen3 (>=4.51) & TRL mới nhất không xung đột
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.51.0', 'peft>=0.14.0', 'trl>=0.14.0', 'bitsandbytes>=0.45.0',
    'datasets', 'accelerate', 'huggingface_hub', 'safetensors', 'sentencepiece'
])

# 3. Đưa `src/` vào sys.path ngay trong kernel đang chạy (vì Jupyter không tự đọc lại file `.pth` mới khi chưa restart kernel!)
src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()

import huggingface_hub
import peft
import transformers
import trl
import vn_rsi  # noqa: F401

print(
    f'✅ Environment Ready! vn_rsi={Path(vn_rsi.__file__).parent} | '
    f'transformers={transformers.__version__} | peft={peft.__version__} | '
    f'trl={trl.__version__} | hf_hub={huggingface_hub.__version__}'
)

In [ ]:
# Cell 4: Kiểm tra Anti-Leakage, Shift-Invariance & Sandbox Seal trước khi Train
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_sandbox_seal.py', 'tests/test_maveq_v4.py', 'tests/test_veteran_reasoning.py', '-q'
], check=True)

In [ ]:
# Cell 5: Cấu hình Huấn luyện Open-Source Model & Hugging Face Repo của bạn (Chỉnh tại đây)
import os

# --- 1. Cấu hình Mô hình & Huấn luyện ---
PRESET = 'auto'                    # 'auto' -> kaggle_t4_optimal (Colab 1xT4) | kaggle_max_100m (Kaggle 2xT4)
MODE = 'maveq_v4'                  # 'maveq_v4' | 'full'
SLM_MODEL = 'Qwen/Qwen3-1.7B'      # Đổi sang 'Qwen/Qwen2.5-1.5B-Instruct', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', hoặc 'Qwen/Qwen3-4B'
ENABLE_VETERAN_DPO = True          # Bật train 2-Stage 4-bit QLoRA SFT -> DPO trên GPU
SFT_EPOCHS = 2                     # Số epoch Stage-1 SFT (5-step Veteran CIO)
DPO_EPOCHS = 1                     # Số epoch Stage-2 DPO (Veteran vs. FOMO)
EPOCHS = 10                        # Số epoch diffu-GRPO cho Tree-MDLM mỗi vòng
ROUNDS = 6                         # Số vòng tiến hóa MAVE-Q v4
SMOKE_TEST = False                 # Đặt True nếu muốn chạy nháp siêu nhanh (~1-2 phút) để test luồng

# --- 2. Cấu hình Push lên Hugging Face của bạn ---
PUSH_TO_HF = True                          # Đặt True để tự động đẩy model lên Hugging Face sau khi train
HF_REPO_ID = 'maveq-v4-qwen3-veteran'      # Chỉ cần ghi 'ten-repo' (tự gắn username HF của bạn) hoặc 'username/ten-repo'
HF_PRIVATE = False                         # False = Public repo | True = Private repo
HF_MERGE_16BIT = False                     # False = Push LoRA+DPO Adapter nhẹ (~30MB) | True = Merge thành Full FP16 Model (~3.4GB)
HF_DATASET_REPO = ''                       # Tùy chọn: điền 'maveq-v4-veteran-dpo-data' nếu muốn push luôn tập dữ liệu SFT+DPO

CKPT_DIR = '/kaggle/working/maveq_v4_ckpts' if os.path.exists('/kaggle/working') else 'artifacts/checkpoints'
print('Checkpoints dir:', CKPT_DIR)
print('Open-Source SLM:', SLM_MODEL, '| Push to HF:', PUSH_TO_HF, '->', HF_REPO_ID)

In [ ]:
# Cell 6: Xác thực Hugging Face Token (Tự đọc Colab/Kaggle Secret `HF_TOKEN` hoặc nhập ẩn nếu chưa có)
import getpass
import os
import sys
from pathlib import Path

src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from huggingface_hub import HfApi
from vn_rsi.veteran.hf_hub_pusher import resolve_hf_token

hf_token = resolve_hf_token()
if PUSH_TO_HF and not hf_token:
    hf_token = getpass.getpass('🔑 Nhập Hugging Face Write Token (hf_...): ').strip()
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    who = HfApi(token=hf_token).whoami()
    print(f"✅ Đã kết nối Hugging Face thành công! Tài khoản: @{who.get('name')}")
else:
    print('ℹ️ Chưa cung cấp HF_TOKEN (bỏ qua bước push lên Hugging Face).')

In [ ]:
# Cell 7: Huấn luyện trọn bộ MAVE-Q v4 Core + Open-Source Model (4-bit QLoRA SFT -> DPO)
import json
import subprocess
import sys
from pathlib import Path
import torch

cmd = [
    sys.executable, 'train_colab_kaggle.py',
    '--preset', PRESET,
    '--mode', MODE,
    '--slm-model', SLM_MODEL,
    '--sft-epochs', str(SFT_EPOCHS),
    '--dpo-epochs', str(DPO_EPOCHS),
    '--epochs', str(EPOCHS),
    '--rounds', str(ROUNDS),
    '--checkpoint-dir', CKPT_DIR,
    '--resume',
]
if ENABLE_VETERAN_DPO:
    cmd.append('--enable-veteran-dpo')
    if torch.cuda.is_available():
        cmd.append('--strict-slm')  # Báo lỗi rõ ràng ngay nếu GPU QLoRA/DPO gặp sự cố
if SMOKE_TEST:
    cmd.append('--smoke-test')

print('🚀 Running command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

report_file = Path(CKPT_DIR) / 'training_report.json'
train_report = json.loads(report_file.read_text(encoding='utf-8'))
print('\n📊 Kết quả huấn luyện Open-Source SLM (Stage 4):', json.dumps(train_report.get('veteran_qlora_dpo', {}), indent=2))

In [ ]:
# Cell 8: Push Model đã Train (LoRA/DPO Adapter hoặc Full FP16) + MAVE-Q v4 Core lên Hugging Face của bạn!
import json
import sys
from pathlib import Path

src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from vn_rsi.veteran.hf_hub_pusher import push_trained_veteran_to_hub

pushed_info = None
if PUSH_TO_HF:
    pushed_info = push_trained_veteran_to_hub(
        checkpoint_dir=CKPT_DIR,
        hf_repo_id=HF_REPO_ID,
        private=HF_PRIVATE,
        merge_16bit=HF_MERGE_16BIT,
        base_model_id=SLM_MODEL,
        hf_dataset_repo=HF_DATASET_REPO or None,
    )
    print('🎉 ĐÃ PUSH THÀNH CÔNG LÊN HUGGING FACE CỦA BẠN!')
    print(json.dumps(pushed_info, indent=2, ensure_ascii=False))
    print('🔗 Link Model trên Hugging Face:', pushed_info['model_url'])
    if 'dataset_url' in pushed_info:
        print('🔗 Link Dataset trên Hugging Face:', pushed_info['dataset_url'])
else:
    print('PUSH_TO_HF=False -> Bỏ qua upload. Đổi PUSH_TO_HF=True rồi chạy lại Cell này bất kỳ lúc nào!')

In [ ]:
# Cell 9: Test hỏi đáp thực tế với Chuyên gia 10 năm + Sinh câu trả lời trực tiếp từ Open-Source Model vừa Train
import subprocess
import sys

for sym, q in [
    ('FPT', 'Đánh giá cơ hội mua FPT và rủi ro T+2.5 hiện tại?'),
    ('NVL_STRESS', 'Có nên bắt đáy NVL_STRESS khi cổ phiếu giảm mạnh không?'),
]:
    subprocess.run([
        sys.executable, 'scripts/ask_veteran_agent.py',
        '--symbol', sym,
        '--checkpoint-dir', CKPT_DIR,
        '--slm-base-model', SLM_MODEL,
        '--use-slm-gen',
        '--question', q,
    ], check=True)

In [ ]:
# Cell 10 (Tùy chọn): Kiểm chứng tải ngược Model từ chính Hugging Face Repo của bạn về chạy Inference
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if pushed_info and torch.cuda.is_available():
    repo_id = pushed_info['model_repo_id']
    print(f'🔄 Đang test kéo model trực tiếp từ Hugging Face: {repo_id} ...')
    tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    if pushed_info.get('uploaded_merged_16bit'):
        hf_model = AutoModelForCausalLM.from_pretrained(
            repo_id, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
        )
    else:
        base = AutoModelForCausalLM.from_pretrained(
            SLM_MODEL, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
        )
        hf_model = PeftModel.from_pretrained(base, repo_id)
    hf_model.eval()
    sample_prompt = (
        '[Date=2024-06-28 | Symbol=FPT (HOSE)] Mkt20d=+3.4%, Breadth=62%, VolRatio=0.94, '
        'TraumaProb=14%, Stock3d=+1.8%, Stock20d=+8.5%, TShock=1.25x, Foreign5d=+4.2%, '
        'PIT_Announced=True, DistFloor=6.1%, DistCeil=4.8%. '
        'Hãy phân tích theo quy trình 5 bước của Chuyên gia Đầu tư 10 năm kinh nghiệm.'
    )
    text_in = tok.apply_chat_template([{'role': 'user', 'content': sample_prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text_in, return_tensors='pt').to(hf_model.device)
    with torch.no_grad():
        out = hf_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    print('\n🤖 Output từ Model trên Hugging Face của bạn:\n')
    print(tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))